In [1]:
import sys
import os
import time
import math

SRC_PATH = os.path.normpath(os.path.join(os.path.abspath(''), '..', 'src'))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model
from solvers.model import ModelSolver
from solvers.FRG import FRGSolver
from settings import INSTANCE_FOLDER

In [2]:
model = load_model(CPMPTransformer, 'v9_dataBSG_250k')
print(f"Modelo v9_dataBSG_250k cargado — H={model.hyperparams['H']}")

class SafeModelSolver(ModelSolver):
    def solve_from_path(self, instance_path, H, max_steps):
        try:
            return super().solve_from_path(instance_path, H, max_steps)
        except (ValueError, RuntimeError, IndexError):
            return False, max_steps

solver_model  = SafeModelSolver(model)  # BSG250k
solver_greedy = FRGSolver(beams=0)      # Greedy puro (FRG --no-assignement)

print("Solvers listos:")
print(f"  - BSG250k  : {solver_model.name}")
print(f"  - Greedy   : {solver_greedy.name} (beams=0)")

Modelo v9_dataBSG_250k cargado — H=12
Solvers listos:
  - BSG250k  : ModelSolver
  - Greedy   : FRG (beams=0)


In [3]:
def benchmark_solver(solver, dat_files, H, max_steps):
    solved_list, steps_list, time_list = [], [], []
    for filepath in dat_files:
        t0 = time.perf_counter()
        solved, steps = solver.solve_from_path(filepath, H, max_steps)
        elapsed = time.perf_counter() - t0
        solved_list.append(solved)
        steps_list.append(steps)
        time_list.append(elapsed)
    return solved_list, steps_list, time_list


def avg(values, mask=None):
    if mask is not None:
        vals = [v for v, ok in zip(values, mask) if ok]
    else:
        vals = list(values)
    return sum(vals) / len(vals) if vals else math.nan


def pct_better_quality(avg_m, avg_g):
    """Retorna (ganador, porcentaje) comparando avg pasos — menos es mejor."""
    if math.isnan(avg_m) or math.isnan(avg_g):
        return None, math.nan
    if avg_m < avg_g:
        return 'Modelo', (avg_g - avg_m) / avg_g * 100
    elif avg_g < avg_m:
        return 'Greedy', (avg_m - avg_g) / avg_m * 100
    return 'Empate', 0.0


def pct_faster(avg_tm, avg_tg):
    """Retorna (ganador, porcentaje) comparando avg tiempo — menos es mejor."""
    if math.isnan(avg_tm) or math.isnan(avg_tg):
        return None, math.nan
    if avg_tm < avg_tg:
        return 'Modelo', (avg_tg - avg_tm) / avg_tg * 100
    elif avg_tg < avg_tm:
        return 'Greedy', (avg_tm - avg_tg) / avg_tm * 100
    return 'Empate', 0.0

In [4]:
CVS_PATH = INSTANCE_FOLDER / 'benchmarks' / 'CVS'
MAX_STEPS = 100

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

results = {}

print(f"Corriendo benchmark en {len(cvs_folders)} categorías CVS...\n")

for folder_name in cvs_folders:
    H_real, S_real = [int(x) for x in folder_name.split('-')]
    H_inf = H_real + 2  # convención CVS
    folder_path = CVS_PATH / folder_name

    dat_files = sorted([
        str(folder_path / f)
        for f in os.listdir(folder_path)
        if f.endswith('.dat')
    ])

    print(f"  [{folder_name}] {len(dat_files)} instancias — H={H_inf}, S={S_real}, max_steps={MAX_STEPS}")

    solved_m, steps_m, time_m = benchmark_solver(solver_model,  dat_files, H_inf, MAX_STEPS)
    solved_g, steps_g, time_g = benchmark_solver(solver_greedy, dat_files, H_inf, MAX_STEPS)

    results[folder_name] = {
        'H': H_real, 'S': S_real, 'n': len(dat_files),
        'solved_m': solved_m, 'steps_m': steps_m, 'time_m': time_m,
        'solved_g': solved_g, 'steps_g': steps_g, 'time_g': time_g,
    }

print("\nBenchmark completado.")

Corriendo benchmark en 21 categorías CVS...

  [10-10] 40 instancias — H=12, S=10, max_steps=100


/Users/felipeastudillo/Desktop/Proyectos/Universidad/CPMP-Transformer/CPMP-Transformer/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


  [10-6] 40 instancias — H=12, S=6, max_steps=100
  [3-3] 40 instancias — H=5, S=3, max_steps=100
  [3-4] 40 instancias — H=5, S=4, max_steps=100
  [3-5] 40 instancias — H=5, S=5, max_steps=100
  [3-6] 40 instancias — H=5, S=6, max_steps=100
  [3-7] 40 instancias — H=5, S=7, max_steps=100
  [3-8] 40 instancias — H=5, S=8, max_steps=100
  [4-4] 40 instancias — H=6, S=4, max_steps=100
  [4-5] 40 instancias — H=6, S=5, max_steps=100
  [4-6] 40 instancias — H=6, S=6, max_steps=100
  [4-7] 40 instancias — H=6, S=7, max_steps=100
  [5-10] 40 instancias — H=7, S=10, max_steps=100
  [5-4] 40 instancias — H=7, S=4, max_steps=100
  [5-5] 40 instancias — H=7, S=5, max_steps=100
  [5-6] 40 instancias — H=7, S=6, max_steps=100
  [5-7] 40 instancias — H=7, S=7, max_steps=100
  [5-8] 40 instancias — H=7, S=8, max_steps=100
  [5-9] 40 instancias — H=7, S=9, max_steps=100
  [6-10] 40 instancias — H=8, S=10, max_steps=100
  [6-6] 40 instancias — H=8, S=6, max_steps=100

Benchmark completado.


In [5]:
# ─── Tabla por categoría ────────────────────────────────────────────────────
W = 160
SEP = '─' * W
HDR_SEP = '═' * W

col_hdr = (
    f"{'Categoría':>10}  {'H':>3} {'S':>3} {'N':>4}  "
    f"{'── BSG250k ──':^38}  "
    f"{'── Greedy ──':^38}  "
    f"{'Calidad':^22}  {'Velocidad':^22}"
)
sub_hdr = (
    f"{'':>10}  {'':>3} {'':>3} {'':>4}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'ganador':^22}  {'ganador':^22}"
)

print(HDR_SEP)
print(col_hdr)
print(sub_hdr)
print(HDR_SEP)

for folder_name in cvs_folders:
    r = results[folder_name]
    n = r['n']

    nm = sum(r['solved_m'])
    ng = sum(r['solved_g'])

    # Máscara instancias resueltas por ambos (para comparación justa de calidad)
    both = [sm and sg for sm, sg in zip(r['solved_m'], r['solved_g'])]

    avg_steps_m      = avg(r['steps_m'], r['solved_m'])
    avg_steps_g      = avg(r['steps_g'], r['solved_g'])
    avg_steps_m_both = avg(r['steps_m'], both)
    avg_steps_g_both = avg(r['steps_g'], both)

    avg_time_m = avg(r['time_m'])
    avg_time_g = avg(r['time_g'])

    q_winner, q_pct = pct_better_quality(avg_steps_m_both, avg_steps_g_both)
    s_winner, s_pct = pct_faster(avg_time_m, avg_time_g)

    def fmt_steps(a): return f"{a:>8.1f}" if not math.isnan(a) else "       -"
    def fmt_time(a):  return f"{a:>10.4f}s" if not math.isnan(a) else "          -"
    def fmt_winner(w, p):
        if w is None or math.isnan(p): return "       -"
        if w == 'Empate': return "     Empate"
        return f"{w} +{p:5.1f}%"

    print(
        f"{folder_name:>10}  {r['H']:>3} {r['S']:>3} {n:>4}  "
        f"{nm:>3}/{n:<3} ({nm/n:>5.1%}) {fmt_steps(avg_steps_m)} {fmt_time(avg_time_m)}  "
        f"{ng:>3}/{n:<3} ({ng/n:>5.1%}) {fmt_steps(avg_steps_g)} {fmt_time(avg_time_g)}  "
        f"{fmt_winner(q_winner, q_pct):^22}  {fmt_winner(s_winner, s_pct):^22}"
    )

print(HDR_SEP)

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Categoría    H   S    N              ── BSG250k ──                            ── Greedy ──                      Calidad                Velocidad       
                            resuelto  avg pasos   avg tiempo    resuelto  avg pasos   avg tiempo         ganador                 ganador        
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
     10-10   10  10   40    0/40  ( 0.0%)        -     0.1536s    0/40  ( 0.0%)        -     0.0036s                -             Greedy + 97.7%    
      10-6   10   6   40    0/40  ( 0.0%)        -     0.1310s    0/40  ( 0.0%)        -     0.0048s                -             Greedy + 96.4%    
       3-3    3   3   40   40/40  (100.0%)     10.2     0.0116s   40/40  (100.0%) 

In [6]:
# ─── Resumen global ─────────────────────────────────────────────────────────
all_solved_m = [s for r in results.values() for s in r['solved_m']]
all_steps_m  = [s for r in results.values() for s in r['steps_m']]
all_time_m   = [t for r in results.values() for t in r['time_m']]

all_solved_g = [s for r in results.values() for s in r['solved_g']]
all_steps_g  = [s for r in results.values() for s in r['steps_g']]
all_time_g   = [t for r in results.values() for t in r['time_g']]

total_n  = len(all_solved_m)
total_nm = sum(all_solved_m)
total_ng = sum(all_solved_g)

both_global = [sm and sg for sm, sg in zip(all_solved_m, all_solved_g)]
global_avg_steps_m      = avg(all_steps_m, all_solved_m)
global_avg_steps_g      = avg(all_steps_g, all_solved_g)
global_avg_steps_m_both = avg(all_steps_m, both_global)
global_avg_steps_g_both = avg(all_steps_g, both_global)

global_avg_time_m = avg(all_time_m)
global_avg_time_g = avg(all_time_g)

q_winner_g, q_pct_g = pct_better_quality(global_avg_steps_m_both, global_avg_steps_g_both)
s_winner_g, s_pct_g = pct_faster(global_avg_time_m, global_avg_time_g)

def quality_line(w, pct, avg_m, avg_g):
    if w == 'Empate':
        return f"Empate — ambos promedian {avg_m:.2f} pasos"
    return f"{w} es {pct:.1f}% mejor en calidad ({avg_m:.2f} vs {avg_g:.2f} pasos)"

def speed_line(w, pct, avg_tm, avg_tg):
    if w == 'Empate':
        return f"Empate — ambos promedian {avg_tm:.4f}s"
    return f"{w} es {pct:.1f}% más rápido ({avg_tm:.4f}s vs {avg_tg:.4f}s por instancia)"

BW = 56
print('═' * BW)
print(f"{'  RESUMEN GLOBAL':^{BW}}")
print('═' * BW)
print(f"  Instancias totales  : {total_n}")
print(f"  BSG250k  resueltas  : {total_nm}/{total_n} ({total_nm/total_n:6.2%})")
print(f"  Greedy   resueltas  : {total_ng}/{total_n} ({total_ng/total_n:6.2%})")
print()
print(f"  Avg pasos (propios resueltos)")
print(f"    BSG250k : {global_avg_steps_m:.2f} pasos")
print(f"    Greedy  : {global_avg_steps_g:.2f} pasos")
print(f"  Avg pasos (instancias resueltas por ambos)")
print(f"    BSG250k : {global_avg_steps_m_both:.2f} pasos")
print(f"    Greedy  : {global_avg_steps_g_both:.2f} pasos")
print(f"  Calidad  : {quality_line(q_winner_g, q_pct_g, global_avg_steps_m_both, global_avg_steps_g_both)}")
print()
print(f"  Avg tiempo por instancia")
print(f"    BSG250k : {global_avg_time_m:.4f}s")
print(f"    Greedy  : {global_avg_time_g:.4f}s")
print(f"  Velocidad: {speed_line(s_winner_g, s_pct_g, global_avg_time_m, global_avg_time_g)}")
print('═' * BW)

════════════════════════════════════════════════════════
                      RESUMEN GLOBAL                    
════════════════════════════════════════════════════════
  Instancias totales  : 840
  BSG250k  resueltas  : 757/840 (90.12%)
  Greedy   resueltas  : 760/840 (90.48%)

  Avg pasos (propios resueltos)
    BSG250k : 30.29 pasos
    Greedy  : 31.29 pasos
  Avg pasos (instancias resueltas por ambos)
    BSG250k : 30.29 pasos
    Greedy  : 31.16 pasos
  Calidad  : Modelo es 2.8% mejor en calidad (30.29 vs 31.16 pasos)

  Avg tiempo por instancia
    BSG250k : 0.0490s
    Greedy  : 0.0037s
  Velocidad: Greedy es 92.5% más rápido (0.0490s vs 0.0037s por instancia)
════════════════════════════════════════════════════════
